In [1]:
import pandas as pd
import numpy as np
import requests 
import re
import spacy
import warnings
warnings.filterwarnings("ignore")

In [2]:
overview = []
Id = []
title = []
original_title = []
original_language = []


In [3]:
def extract(Id,page, overview, title, original_language, original_title):
    for j in range(1,244):
        response = requests.get(
            "https://api.themoviedb.org/3/movie/now_playing?api_key=7419e8710a99cc6323c2bc6a406c6994&language=en-US&page={}".format(j)
        )
        data = response.json()

        
        for i in data.get("results", []):
            Id.append(i.get("id",""))
            overview.append(i.get("overview", ""))            
            title.append(i.get("title", ""))
            original_language.append(i.get("original_language", ""))
            original_title.append(i.get("original_title", ""))

In [4]:
extract(Id,244,overview,title,original_language,original_title)

In [5]:
movies = pd.DataFrame({
    "id" : Id,
    "overview" : overview,
    "title" : title,
    "original_title" : original_title,
    "original_language" : original_language
})

In [6]:
movies.to_csv("Now playing movies.csv")

In [7]:
movies.shape

(4858, 5)

In [8]:
data = movies.drop_duplicates()

In [9]:
data.shape

(4599, 5)

In [10]:
data.duplicated().sum()

0

In [11]:
data.drop(columns=['id'],inplace=True)

In [12]:
data.sample(3)

,overview,title,original_title,original_language
2940,"After marrying her wife in Switzerland, Chines...",My Dear Dear Home,回乡偶记,zh
808,"One night, a woman in danger calls the police....",Through the Night,Quitter la nuit,fr
4264,An electric transition. The void left by a vic...,C.R.Y.S.T.A.L.,C.R.Y.S.T.A.L.,en


# Lower case

In [13]:
data["overview"] = data["overview"].str.lower()
data["title"] = data["title"].str.lower()
data["original_language"] = data["original_language"].str.lower()
data["original_title"] = data["original_title"].str.lower()

In [14]:
data["title"][4633]

'в рождество все немного волхвы...'

# Removing HTML Tags

In [15]:
import re

def removing_html_tags(text):
    if isinstance(text,str):
        pattern = re.compile("<.*?>")
        return pattern.sub("",text)
        
    return text

In [16]:
removing_html_tags("<a href=https:localhost:8888.notebooks>")

''

In [17]:
# IN data set there are not html tage


# Remove Punctuation

In [18]:
import string

exclude = string.punctuation

In [19]:
def remove_puns(text):
    if isinstance(text,str):
        return text.translate(str.maketrans("","",exclude))

    return text

In [20]:
data["overview"] = data["overview"].apply(remove_puns)
data["title"] = data["title"].apply(remove_puns)
data["original_language"] = data['original_language'].apply(remove_puns)
data["original_title"] = data['original_title'].apply(remove_puns)

# Spelling correction

In [21]:
from textblob import TextBlob

def spelling_correction(text):
    if isinstance(text, str):
        return str(TextBlob(text).correct())
    return text

In [43]:
spelling_correction("Solor")

'Color'

In [23]:
data["original_language"] = data["original_language"].apply(spelling_correction)
data["original_language"]

0       re
1       en
2       en
3       en
4       en
        ..
4853    re
4854    re
4855    re
4856    en
4857    en
Name: original_language, Length: 4599, dtype: object

In [24]:
# data["overview"] = data["overview"].apply(spelling_correction) # it taking too long computation time i don't apply on the whole dataset because they also can porive hte incorrect spellign or change the another words

In [25]:
data["overview"][10]

'having found the safety of the greenland bunker after the comet clarke decimated the earth the garrity family must now risk everything to embark on a perilous journey across the wasteland of europe to find a new home'

# Removign Stop words

In [26]:
from nltk.corpus import stopwords

sw = stopwords.words('english')
sw

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [27]:
def rem_stop_words(text):
    if isinstance(text,str):
       return " ".join([i for i in text.split() if i not in sw])
    return text

In [28]:
rem_stop_words("i am amit")

'amit'

In [29]:
# applying in the dataset 

data["overview"] = data["overview"].apply(rem_stop_words)

data["title"] = data["title"].apply(rem_stop_words)

data["original_language"] = data["original_language"].apply(rem_stop_words)

data["original_title"] = data["original_title"].apply(rem_stop_words)

# Tokenization

In [30]:
# from nltk.tokenize import sent_tokenize,word_tokenize

# def my_word_tokenize(text):
#     if isinstance(text,str):
#         return " ".join(word_tokenize(text))

#     return text

# or

import spacy

sy = spacy.load("en_core_web_sm")

def w_t(text):
    if isinstance(text,str):
        tem = sy(text)
        return [word.text for word in tem]
    return text

In [31]:
c = (w_t("hi my name is amit"))
c

['hi', 'my', 'name', 'is', 'amit']

In [32]:
data["overview"] = data["overview"].apply(w_t)

data["title"] = data["title"].apply(w_t)

data["original_language"] = data["original_language"].apply(w_t)

data["original_title"] = data["original_title"].apply(w_t)

In [33]:
data.head(3)

,overview,title,original_title,original_language
0,"[high, school, student, polina, saved, bullyin...","[heart, broken]","[твоё, сердце, будет, разбито]",[]
1,"[elusive, thief, whose, highstakes, heists, un...","[crime, 101]","[crime, 101]",[en]
2,"[two, gangsters, woman, love, try, survive, da...","[mike, nick, nick, alice]","[mike, nick, nick, alice]",[en]


# stemming

In [34]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

def stemming(text):
    if isinstance(text, str):
        return " ".join([ps.stem(word) for word in text.split()])
    return text

In [35]:
stemming("I am loving learning NLP")

'i am love learn nlp'

In [36]:
data["overview"] = data["overview"].apply(stemming)

data["title"] = data["title"].apply(stemming)

data["original_language"] = data["original_language"].apply(stemming)

data["original_title"] = data["original_title"].apply(stemming)

In [37]:
data.head()

,overview,title,original_title,original_language
0,"[high, school, student, polina, saved, bullyin...","[heart, broken]","[твоё, сердце, будет, разбито]",[]
1,"[elusive, thief, whose, highstakes, heists, un...","[crime, 101]","[crime, 101]",[en]
2,"[two, gangsters, woman, love, try, survive, da...","[mike, nick, nick, alice]","[mike, nick, nick, alice]",[en]
3,"[man, living, selfimposed, exile, remote, isla...",[shelter],[shelter],[en]
4,"[scientists, discovered, hop, human, conscious...",[hoppers],[hoppers],[en]
